In [1]:
!pip install torch transformers graphviz

In [2]:
import random
from dataclasses import dataclass, field
from typing import Any, List, Tuple, Optional, Dict

random.seed(42)

@dataclass(frozen=True)
class Step:
    action: Any
    state: Any
    cost: float

class SearchProblem:

    def start_state(self) -> Any:
        raise NotImplementedError

    def successors(self, state: Any) -> List[Step]:
        raise NotImplementedError
    
    def is_end(self, state: Any) -> bool:
        raise NotImplementedError

@dataclass
class Solution:
    steps: List[Step]
    cost: float = field(init=False)

    def __post_init__(self):
        self.cost = sum(step.cost for step in self.steps)

step1 = Step(action="a", state=1, cost=1)
step2 = Step(action="b", state=2, cost=3)
solution = Solution([step1, step2])
solution.cost


4

In [3]:
# Example 1 - Travel Search Problem
# Move from 1 to N, action could be walk（+1，Cost=1）or  tram（*2，Cost=2）

class TravelSearchProblem(SearchProblem):

    def __init__(self, num_locs: int):
        self.num_locs = num_locs

    def start_state(self) -> int:
        return 1
    
    def successors(self, state: int) -> List[Step]:
        steps = []
        if state < self.num_locs:
            steps.append(Step(action="walk", state=state + 1, cost=1.0))
        if state * 2 <= self.num_locs:
            steps.append(Step(action="tram", state=state * 2, cost=2.0))
        return steps
        
    def is_end(self, state: int) -> bool:
        return state == self.num_locs

# Quick test
problem = TravelSearchProblem(num_locs=10)
print(f"Start State:{problem.start_state()}")
print(f"Successors from state 1:{problem.successors(1)}")

Start State:1
Successors from state 1:[Step(action='walk', state=2, cost=1.0), Step(action='tram', state=2, cost=2.0)]


In [4]:
# Example 2 - Limited Travel Search Problem 
# Add constrait of limited tickets to take the tram

@dataclass(frozen=True)
class TravelSearchState:
    loc: int
    tickets: int #remaining tickets


class LimitedTravelSearchProblem(SearchProblem):

    def __init__(self, num_locs: int, num_tickets: int):
        self.num_locs = num_locs
        self.num_tickets = num_tickets

    def start_state(self) -> TravelSearchState:
        return TravelSearchState(loc=1, tickets=self.num_tickets)
    
    def successors(self, state: TravelSearchState) -> List[Step]:
        steps = []
        if state.loc < self.num_locs:
            steps.append(Step(
                action="walk", 
                state=TravelSearchState(state.loc + 1, state.tickets), 
                cost=1.0))
        if state.loc * 2 <= self.num_locs and state.tickets > 0:
            steps.append(Step(
                action="tram", 
                state=TravelSearchState(state.loc * 2, state.tickets - 1), 
                cost=2.0))
        return steps
        
    def is_end(self, state: TravelSearchState) -> bool:
        return state.loc == self.num_locs

# Test limited travel problem
limited_problem = LimitedTravelSearchProblem(num_locs=10, num_tickets=1)
print(f"Limited Travel Start:{limited_problem.start_state()}")
print(f"Successors from state 1:{limited_problem.successors(TravelSearchState(1, 1))}")

Limited Travel Start:TravelSearchState(loc=1, tickets=1)
Successors from state 1:[Step(action='walk', state=TravelSearchState(loc=2, tickets=1), cost=1.0), Step(action='tram', state=TravelSearchState(loc=2, tickets=0), cost=2.0)]


In [5]:
# Exhaustive Search

# Computes the exact minimum cost solution by evaluating all paths recursively.
# Returns the optimal solution and the count of states explored.
def exhaustive_search(problem: SearchProblem) -> Tuple[Solution, int]:
    num_explored = 0

    def future_solution(state: Any) -> Solution:
        nonlocal num_explored
        num_explored += 1
        if problem.is_end(state):
            return Solution([])
        successors = problem.successors(state)
        if not successors: # If leaf, return solution with infinite cost to prune out this path
            return Solution(steps=[
                Step(action="dead_end", cost=float('inf'), state=state)
                ])
        solutions = [Solution([s] + future_solution(s.state).steps) for s in successors]
        return min(solutions, key=lambda s: s.cost)
    
    return future_solution(problem.start_state()), num_explored

# Experiment: Scaling behavior of Exhaustive Search
for n in [4, 10, 17]:
    p = TravelSearchProblem(num_locs=n)
    sol, explored = exhaustive_search(p)
    print(f"N={n:2d} | Explored States:{explored:6d} | Min Cost:{sol.cost}")

N= 4 | Explored States:     9 | Min Cost:3.0
N=10 | Explored States:    59 | Min Cost:6.0
N=17 | Explored States:   237 | Min Cost:8.0


In [6]:
# Dynamic Programming

# exhaustive search with cache
# Returns the optimal solution, the count of states explored (not cached), and cache
def dynamic_programming_search(problem: SearchProblem) -> Tuple[Solution, int, dict]:
    num_explored = 0
    cache: Dict[Any, Solution] = {} # store state -> solution

    def future_solution(state: Any) -> Solution:
        if state in cache:
            return cache[state]

        nonlocal num_explored # don't count cached state into num_explored
        num_explored += 1
        
        if problem.is_end(state):
            best_solution = Solution([])
        else:
            successors = problem.successors(state)
            if not successors: # If leaf, return solution with infinite cost to prune out this path
                best_solution = Solution(steps=[
                    Step(action="dead_end", cost=float('inf'), state=state)
                ])
            solutions = [Solution([s] + future_solution(s.state).steps) for s in successors]
            best_solution = min(solutions, key=lambda s: s.cost)
        cache[state] = best_solution
        return best_solution
    return future_solution(problem.start_state()), num_explored, cache

# Experiment: Scaling behavior of dynamic programming Search
for n in [4, 10, 17]:
    p = TravelSearchProblem(num_locs=n)
    sol, explored, _ = dynamic_programming_search(p)
    print(f"N={n:2d} | Explored States:{explored:6d} | Min Cost:{sol.cost}")


N= 4 | Explored States:     4 | Min Cost:3.0
N=10 | Explored States:    10 | Min Cost:6.0
N=17 | Explored States:    17 | Min Cost:8.0


In [7]:
# Best-of-N Search

def uniform_policy(problem: SearchProblem, state: Any) -> Step:
    successors = problem.successors(state)
    return random.choice(successors)

def rollout(problem: SearchProblem, policy, max_steps: int = 10) -> Solution:
    steps = []
    state = problem.start_state()
    for _ in range(max_steps):
        if problem.is_end(state):
            break
        step = policy(problem, state)
        steps.append(step) 
        state = step.state
    return Solution(steps)

def best_of_n(problem: SearchProblem, policy, num_candidates: int, max_steps: int = 10) -> Tuple[Solution, int]:
    num_explored = 0
    solutions = []
    for _ in range(num_candidates):
        solution = rollout(problem, policy, max_steps)
        solutions.append(solution)
        num_explored += len(solution.steps)

    best_solution = min(solutions, key=lambda s: s.cost)
    return best_solution, num_explored

# Test Best-of-N
p = TravelSearchProblem(num_locs=10)
best_sol, explored = best_of_n(p, uniform_policy, num_candidates=20, max_steps=10)
print(f"Best-of-N | Explored Steps:{explored} | Best Cost Found:{best_sol.cost}")

Best-of-N | Explored Steps:116 | Best Cost Found:6.0


In [8]:
# Beam Search

def beam_search(problem: SearchProblem, beam_width: int, max_steps: int = 10) -> Tuple[Optional[Solution], int]:
    num_explored = 0
    candidates = [Solution([])]
    for _ in range(max_steps):
        new_candidates = []
        for candidate in candidates:
            state = candidate.steps[-1].state if candidate.steps else problem.start_state()
            if problem.is_end(state):
                new_candidates.append(candidate)
            else:
                for successor in problem.successors(state):
                    solution = Solution(candidate.steps + [successor])
                    new_candidates.append(solution)
                    num_explored += 1
        new_candidates.sort(key=lambda s: s.cost)
        candidates = new_candidates[: beam_width]

    # Keep only candidates that are done
    candidates = [candidate for candidate in candidates if problem.is_end(candidate.steps[-1].state)]
    return candidates[0] if candidates else None, num_explored

# Test Beam Search
p = TravelSearchProblem(num_locs=10)
beam_sol, explored = beam_search(p, beam_width=2, max_steps=10)
print(f"Beam Search | Explored:{explored} | Best Cost:{beam_sol.cost}")


Beam Search | Explored:20 | Best Cost:6.0


In [39]:
from typing import Any

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

# LLM Integration - Search as Test-time Compute
class LanguageModelSearchProblem(SearchProblem):

    def __init__(self, prompt: str, model_id: str = "Qwen/Qwen2.5-0.5B"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).eval()
        self.prompt = prompt

    def start_state(self) -> str:
        return self.prompt

    def is_end(self, state: str) -> bool:
        return self.is_complete_sentense(state)

    def is_complete_sentense(self, state: str) -> bool:
        return state.rstrip().endswith(")")

    # Verifier: check if expression can be evaluated to a valid number.
    def contains_number(self, state: str) -> bool:
        try:
            eval(state) # Use Python internal eval to try to compute the expression
            return True 
        except:
            return False

    def successors(self, state: str) -> list[Step]:
        input_ids = tokenizer(state, return_tensors="pt")["input_ids"]
        steps = []
        with torch.no_grad():
            logits = model(input_ids).logits[0, -1, :]
            probs = F.softmax(logits, dim=-1)
            topk = torch.topk(probs, k=5)

        for index, prob in zip(topk.indices, topk.values):
            token_str = tokenizer.decode([index.item()])
            cost = -torch.log(prob + + 1e-10).item()
            next_state = state + token_str
            if self.is_complete_sentense(next_state) and self.contains_number(next_state):
                cost -= 100 # Verifier reward (negative cost) if solution condition met
            step = Step(action=token_str, state=next_state, cost=cost)
            steps.append(step)

        return steps

def llm_policy(problem: SearchProblem, state: str) -> Step:
    successors = problem.successors(state)
    costs = torch.tensor([s.cost for s in successors])
    probs = F.softmax(-costs, dim=-1)
    index = torch.multinomial(probs, num_samples=1)[0].item()
    return successors[index]

In [40]:
# Run LLM Best-of-N Search Test
torch.manual_seed(42)
p = LanguageModelSearchProblem(prompt="(3 + 7 *", model_id="gpt2")
print("\n--- Executing LLM Best-of-N Search ---")
solution, num_explored = best_of_n(p, llm_policy, num_candidates=5, max_steps=8)
print("\nBest Generated String:")
print(solution.steps[-1].state if solution.steps else "No solution generated")
print(f"Total Path Cost:{solution.cost:.4f}")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


--- Executing LLM Best-of-N Search ---

Best Generated String:
(3 + 7 * 3)
Total Path Cost:-96.9541
